# Healthcare Utilization Pipeline — Demo

Validates the expanded silver layer (patients, encounters, conditions) across MIMIC-IV (ED + Hospital + ICU) and Synthea sources.

In [1]:
import duckdb
import pandas as pd

DB_PATH = "../data/warehouse/mimic_fhir.duckdb"
con = duckdb.connect(DB_PATH, read_only=True)

## 1. Bronze Inventory

In [2]:
con.execute("""
    SELECT json_extract_string(resource, '$.meta.source') as source,
           resource_type, COUNT(*) as cnt
    FROM bronze.fhir_resources
    GROUP BY source, resource_type
    ORDER BY source, cnt DESC
""").fetchdf()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,source,resource_type,cnt
0,mimic-iv,Observation,14352966
1,mimic-iv,Condition,5655376
2,mimic-iv,MedicationDispense,1586053
3,mimic-iv,Encounter,929499
4,mimic-iv,Patient,299712
5,mimic-iv,Location,39
6,mimic-iv,Organization,1
7,synthea,Observation,43096
8,synthea,Procedure,15674
9,synthea,DiagnosticReport,10150


## 2. Silver Row Counts

In [3]:
for table in ['patients', 'encounters', 'conditions']:
    df = con.execute(f"SELECT source, COUNT(*) as rows FROM silver.{table} GROUP BY source ORDER BY source").fetchdf()
    print(f"--- silver.{table} ---")
    display(df)
    print()

--- silver.patients ---


,source,rows
0,mimic-iv,299712
1,synthea,107



--- silver.encounters ---


,source,rows
0,mimic-iv,929499
1,synthea,5545



--- silver.conditions ---


,source,rows
0,mimic-iv,5655376
1,synthea,3443


## 3. Encounter Classes by Source

MIMIC now includes ED (EMER), hospital (EMER/OBSENC/AMB/SS), and ICU (ACUTE) encounters.

In [4]:
con.execute("""
    SELECT source, encounter_class, COUNT(*) as cnt,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY source), 1) as pct
    FROM silver.encounters
    GROUP BY source, encounter_class
    ORDER BY source, cnt DESC
""").fetchdf()

,source,encounter_class,cnt,pct
0,mimic-iv,EMER,594054,63.9
1,mimic-iv,OBSENC,166151,17.9
2,mimic-iv,ACUTE,73181,7.9
3,mimic-iv,AMB,61882,6.7
4,mimic-iv,SS,34231,3.7
5,synthea,AMB,5187,93.5
6,synthea,EMER,182,3.3
7,synthea,IMP,127,2.3
8,synthea,HH,39,0.7
9,synthea,VR,10,0.2


## 4. New Encounter Fields — Admit Source & Discharge Disposition

In [5]:
print("--- Admit source by encounter class (MIMIC top 10) ---")
display(con.execute("""
    SELECT encounter_class, admit_source, COUNT(*) as cnt
    FROM silver.encounters WHERE source = 'mimic-iv' AND admit_source IS NOT NULL
    GROUP BY encounter_class, admit_source ORDER BY cnt DESC LIMIT 10
""").fetchdf())

print()
print("--- Discharge disposition by encounter class (MIMIC top 10) ---")
display(con.execute("""
    SELECT encounter_class, discharge_disposition, COUNT(*) as cnt
    FROM silver.encounters WHERE source = 'mimic-iv' AND discharge_disposition IS NOT NULL
    GROUP BY encounter_class, discharge_disposition ORDER BY cnt DESC LIMIT 10
""").fetchdf())

--- Admit source by encounter class (MIMIC top 10) ---


,encounter_class,admit_source,cnt
0,EMER,WALK IN,251849
1,EMER,AMBULANCE,155752
2,EMER,EMERGENCY ROOM,130210
3,OBSENC,EMERGENCY ROOM,102249
4,SS,PHYSICIAN REFERRAL,34186
5,OBSENC,PHYSICIAN REFERRAL,33870
6,AMB,PHYSICIAN REFERRAL,28225
7,AMB,TRANSFER FROM HOSPITAL,20872
8,EMER,PHYSICIAN REFERRAL,18682
9,EMER,UNKNOWN,15352



--- Discharge disposition by encounter class (MIMIC top 10) ---


,encounter_class,discharge_disposition,cnt
0,EMER,HOME,322528
1,EMER,ADMITTED,158010
2,EMER,HOME HEALTH CARE,39636
3,AMB,HOME,32239
4,OBSENC,HOME,25371
5,EMER,SKILLED NURSING FACILITY,24239
6,SS,HOME,16917
7,OBSENC,HOME HEALTH CARE,14570
8,SS,HOME HEALTH CARE,11184
9,AMB,HOME HEALTH CARE,10182


## 5. ICU Encounters — Parent Linking & Unit Distribution

ICU encounters (class=ACUTE) are child encounters linked to hospital encounters via `parent_encounter_id`.

In [6]:
# ICU unit distribution (resolve location_id via bronze Location resources)
con.execute("""
    WITH loc_names AS (
        SELECT resource_id, json_extract_string(resource, '$.name') AS unit_name
        FROM bronze.fhir_resources WHERE resource_type = 'Location'
    )
    SELECT l.unit_name, COUNT(*) as icu_stays
    FROM silver.encounters e
    LEFT JOIN loc_names l ON e.location_id = l.resource_id
    WHERE e.encounter_class = 'ACUTE'
    GROUP BY l.unit_name
    ORDER BY icu_stays DESC
""").fetchdf()

,unit_name,icu_stays
0,Medical Intensive Care Unit (MICU),15898
1,Medical/Surgical Intensive Care Unit (MICU/SICU),12733
2,Cardiac Vascular Intensive Care Unit (CVICU),11582
3,Surgical Intensive Care Unit (SICU),11161
4,Trauma SICU (TSICU),8692
5,Coronary Care Unit (CCU),8311
6,Neuro Intermediate,2035
7,Neuro Surgical Intensive Care Unit (Neuro SICU),1762
8,Neuro Stepdown,1007


In [7]:
# Verify ICU → Hospital parent linking
con.execute("""
    SELECT
        icu.resource_id AS icu_encounter_id,
        icu.encounter_class AS icu_class,
        icu.period_start AS icu_start,
        icu.period_end AS icu_end,
        parent.encounter_class AS parent_class,
        parent.period_start AS parent_start,
        parent.period_end AS parent_end,
        parent.admit_source,
        parent.discharge_disposition
    FROM silver.encounters icu
    JOIN silver.encounters parent ON icu.parent_encounter_id = parent.resource_id
    WHERE icu.encounter_class = 'ACUTE'
    LIMIT 5
""").fetchdf()

,icu_encounter_id,icu_class,icu_start,icu_end,parent_class,parent_start,parent_end,admit_source,discharge_disposition
0,3f9de2df-a0e2-582f-bd73-b785feebc34f,ACUTE,2136-12-18 00:06:00,2136-12-21 11:48:29,EMER,2136-12-17 23:09:00,2137-01-02 15:52:00,PHYSICIAN REFERRAL,SKILLED NURSING FACILITY
1,778ca7df-f1f4-5399-8459-04f0ab017a29,ACUTE,2133-11-17 20:20:32,2133-11-20 18:58:23,EMER,2133-11-17 20:19:00,2133-12-05 19:29:00,EMERGENCY ROOM,HOME HEALTH CARE
2,eaf3bc77-e511-5d11-9a05-2f52f452a32c,ACUTE,2135-07-20 02:14:00,2135-07-21 18:21:18,OBSENC,2135-07-20 00:24:00,2135-07-24 14:24:00,EMERGENCY ROOM,SKILLED NURSING FACILITY
3,6b32b4ed-5ac3-594b-8d8c-c86506c53564,ACUTE,2132-09-26 14:12:25,2132-09-27 13:12:36,OBSENC,2132-09-25 14:10:00,2132-10-12 12:30:00,WALK-IN/SELF REFERRAL,HOME HEALTH CARE
4,8443d46c-491f-5f0d-9434-ad6718393459,ACUTE,2185-06-11 06:26:14,2185-06-19 04:48:03,EMER,2185-05-16 19:54:00,2185-06-19 02:13:00,PHYSICIAN REFERRAL,DIED


## 6. Null Rate Check

In [8]:
def null_rates(table, columns):
    cases = ", ".join(
        f"ROUND(100.0 * SUM(CASE WHEN {c} IS NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS {c}"
        for c in columns
    )
    return con.execute(f"SELECT source, COUNT(*) as total, {cases} FROM silver.{table} GROUP BY source").fetchdf()

print("--- silver.patients ---")
display(null_rates('patients', ['gender', 'birth_date', 'race', 'ethnicity', 'deceased_datetime']))
print()
print("--- silver.encounters ---")
display(null_rates('encounters', ['patient_id', 'encounter_class', 'period_start', 'period_end',
                                   'admit_source', 'discharge_disposition', 'service_type',
                                   'priority_code', 'parent_encounter_id', 'location_id']))
print()
print("--- silver.conditions ---")
display(null_rates('conditions', ['patient_id', 'encounter_id', 'clinical_status', 'category',
                                   'code', 'code_system', 'onset_datetime']))

--- silver.patients ---


,source,total,gender,birth_date,race,ethnicity,deceased_datetime
0,synthea,107,0.0,0.0,0.0,0.0,89.7
1,mimic-iv,299712,0.0,0.0,12.1,20.5,90.3



--- silver.encounters ---


,source,total,patient_id,encounter_class,period_start,period_end,admit_source,discharge_disposition,service_type,priority_code,parent_encounter_id,location_id
0,synthea,5545,0.0,0.0,0.0,0.0,100.0,99.5,100.0,100.0,100.0,0.0
1,mimic-iv,929499,0.0,0.0,0.0,0.0,7.9,20.7,53.6,53.6,70.3,45.7



--- silver.conditions ---


,source,total,patient_id,encounter_id,clinical_status,category,code,code_system,onset_datetime
0,synthea,3443,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,mimic-iv,5655376,0.0,0.0,100.0,0.0,0.0,0.0,100.0


## 7. Condition Code Systems

MIMIC uses ICD-9 (hospital) and ICD-10 (ED). Synthea uses SNOMED-CT.

In [9]:
con.execute("""
    SELECT source, code_system, COUNT(*) as cnt,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY source), 1) as pct
    FROM silver.conditions
    GROUP BY source, code_system
    ORDER BY source, cnt DESC
""").fetchdf()

,source,code_system,cnt,pct
0,mimic-iv,http://mimic.mit.edu/fhir/mimic/CodeSystem/mim...,3209892,56.8
1,mimic-iv,http://mimic.mit.edu/fhir/mimic/CodeSystem/mim...,2445484,43.2
2,synthea,http://snomed.info/sct,3443,100.0


## 8. Sample Patient Timeline

A single patient's full healthcare journey: ED → Hospital → ICU encounters with linked conditions.

In [10]:
# Find a patient with ED + Hospital + ICU encounters
sample_patient = con.execute("""
    WITH patient_classes AS (
        SELECT patient_id, COUNT(DISTINCT encounter_class) as n_classes,
               BOOL_OR(encounter_class = 'EMER') as has_ed,
               BOOL_OR(encounter_class = 'ACUTE') as has_icu
        FROM silver.encounters WHERE source = 'mimic-iv'
        GROUP BY patient_id
        HAVING has_ed AND has_icu
        LIMIT 1
    )
    SELECT patient_id FROM patient_classes
""").fetchone()[0]

print(f"Patient: {sample_patient}")
print()

# Demographics
display(con.execute(f"SELECT * FROM silver.patients WHERE resource_id = '{sample_patient}'").fetchdf())
print()

# All encounters for this patient
print("--- Encounter timeline ---")
display(con.execute(f"""
    SELECT encounter_class, period_start, period_end,
           ROUND(EXTRACT(EPOCH FROM (period_end - period_start)) / 3600.0, 1) as hours,
           admit_source, discharge_disposition, service_type, parent_encounter_id
    FROM silver.encounters
    WHERE patient_id = '{sample_patient}' AND source = 'mimic-iv'
    ORDER BY period_start
""").fetchdf())

print()
print("--- Conditions for this patient ---")
display(con.execute(f"""
    SELECT c.code, c.code_system, c.code_display, e.encounter_class, e.period_start
    FROM silver.conditions c
    JOIN silver.encounters e ON c.encounter_id = e.resource_id
    WHERE c.patient_id = '{sample_patient}' AND c.source = 'mimic-iv'
    ORDER BY e.period_start
    LIMIT 15
""").fetchdf())

Patient: 8ec2c0ae-24c1-5447-9c0f-66ecc474e9dc



,resource_id,source,gender,birth_date,deceased_datetime,race,ethnicity
0,8ec2c0ae-24c1-5447-9c0f-66ecc474e9dc,mimic-iv,female,2098-04-29,NaT,White,Not Hispanic or Latino



--- Encounter timeline ---


,encounter_class,period_start,period_end,hours,admit_source,discharge_disposition,service_type,parent_encounter_id
0,EMER,2174-04-29 22:57:00,2174-04-30 21:46:00,22.8,HELICOPTER,ADMITTED,None,6f0197d1-7614-52f8-a5e5-cc0b97d8ba2f
1,EMER,2174-04-30 19:42:00,2174-05-02 12:45:00,41.1,TRANSFER FROM HOSPITAL,HOME,NMED,None
2,ACUTE,2174-04-30 21:46:00,2174-05-02 12:48:12,39.0,None,None,None,6f0197d1-7614-52f8-a5e5-cc0b97d8ba2f



--- Conditions for this patient ---


,code,code_system,code_display,encounter_class,period_start
0,R42,http://mimic.mit.edu/fhir/mimic/CodeSystem/mim...,Dizziness and giddiness,EMER,2174-04-29 22:57:00
1,I629,http://mimic.mit.edu/fhir/mimic/CodeSystem/mim...,"Nontraumatic intracranial hemorrhage, unspecified",EMER,2174-04-29 22:57:00
2,R51,http://mimic.mit.edu/fhir/mimic/CodeSystem/mim...,Headache,EMER,2174-04-29 22:57:00
3,I618,http://mimic.mit.edu/fhir/mimic/CodeSystem/mim...,Other nontraumatic intracerebral hemorrhage,EMER,2174-04-30 19:42:00
4,M5136,http://mimic.mit.edu/fhir/mimic/CodeSystem/mim...,"Other intervertebral disc degeneration, lumbar...",EMER,2174-04-30 19:42:00
5,R29702,http://mimic.mit.edu/fhir/mimic/CodeSystem/mim...,NIHSS score 2,EMER,2174-04-30 19:42:00
6,I10,http://mimic.mit.edu/fhir/mimic/CodeSystem/mim...,Essential (primary) hypertension,EMER,2174-04-30 19:42:00
7,H538,http://mimic.mit.edu/fhir/mimic/CodeSystem/mim...,Other visual disturbances,EMER,2174-04-30 19:42:00


## 9. Silver Layer — Complete Feature Inventory

In [11]:
for table in ['patients', 'encounters', 'conditions']:
    print(f"=== silver.{table} ===")
    cols = con.execute(f"DESCRIBE silver.{table}").fetchdf()
    display(cols[['column_name', 'column_type']])
    print()

=== silver.patients ===


,column_name,column_type
0,resource_id,VARCHAR
1,source,VARCHAR
2,gender,VARCHAR
3,birth_date,DATE
4,deceased_datetime,TIMESTAMP
5,race,VARCHAR
6,ethnicity,VARCHAR



=== silver.encounters ===


,column_name,column_type
0,resource_id,VARCHAR
1,source,VARCHAR
2,patient_id,VARCHAR
3,encounter_class,VARCHAR
4,status,VARCHAR
5,period_start,TIMESTAMP
6,period_end,TIMESTAMP
7,type_code,VARCHAR
8,type_display,VARCHAR
9,type_system,VARCHAR



=== silver.conditions ===


,column_name,column_type
0,resource_id,VARCHAR
1,source,VARCHAR
2,patient_id,VARCHAR
3,encounter_id,VARCHAR
4,clinical_status,VARCHAR
5,verification_status,VARCHAR
6,category,VARCHAR
7,code,VARCHAR
8,code_system,VARCHAR
9,code_display,VARCHAR


In [12]:
con.close()